In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

engage_2_value_from_clicks_to_conversions_path = kagglehub.competition_download('engage-2-value-from-clicks-to-conversions')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Imports and Dependencies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, StackingRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
import category_encoders as ce
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import RFE
from seaborn import pairplot

# Loading Dataset

In [ ]:
train = pd.read_csv('/kaggle/input/engage-2-value-from-clicks-to-conversions/train_data.csv')
test = pd.read_csv('/kaggle/input/engage-2-value-from-clicks-to-conversions/test_data.csv')

# Exploratory Data Analysis (EDA)

### Overview of dataset structure and missing values

In [ ]:
train.info()

### Descriptive Analysis

In [ ]:
train.describe()

### Seperating target and feature columns

In [ ]:
y = train["purchaseValue"]
X = train.drop(columns=["purchaseValue"])
X_test = test.copy()

### Analysing categorical features

In [ ]:
print("\n Categorical Feature Counts:")
cat_cols = X.select_dtypes(include='object').columns
for col in cat_cols:
    print(f"\n{col} - Unique: {X[col].nunique()}")
    print(X[col].value_counts(dropna=False).head())

### Visualising target distribution

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(y, kde=True, bins=30)
plt.title("Target Variable Distribution")
plt.show()

### Visualising some numeric feature distributions

In [ ]:
num_cols = X.select_dtypes(include=np.number).columns[:6]

X[num_cols].hist(figsize=(15, 10), bins=30, edgecolor='black')
plt.suptitle("Distributions of Top Numeric Features")
plt.show()

### Pairplot of top correlated numeric features with target

In [ ]:

corr_with_target = X.select_dtypes(include=np.number).corrwith(y).sort_values(key=abs, ascending=False)
top_corr_feats = corr_with_target[1:4].index.tolist()
pairplot_data = X[top_corr_feats].copy()
pairplot_data['target'] = y

sns.pairplot(pairplot_data)
plt.suptitle("Pairwise Relationships", y=1.02)
plt.show()

# Data Preprocessing

### Dividing Numerical and Categorical Columns

In [ ]:
cat_cols = X.select_dtypes(include=["object", "bool"]).columns.tolist()
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

### Impute missing numeric features with median

In [ ]:
num_imputer = SimpleImputer(strategy="median")

X[num_cols] = num_imputer.fit_transform(X[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])

### Impute missing categorical features with most frequent

In [ ]:
cat_imputer = SimpleImputer(strategy="most_frequent")

X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

# Feature Engineering

In [ ]:
city_avg = train.groupby("geoNetwork.city")["purchaseValue"].mean()
X["city_avg_spend"] = X["geoNetwork.city"].map(city_avg).fillna(train["purchaseValue"].mean())
X_test["city_avg_spend"] = X_test["geoNetwork.city"].map(city_avg).fillna(train["purchaseValue"].mean())

region_avg = train.groupby("geoNetwork.region")["purchaseValue"].mean()
X["region_avg_spend"] = X["geoNetwork.region"].map(region_avg).fillna(train["purchaseValue"].mean())
X_test["region_avg_spend"] = X_test["geoNetwork.region"].map(region_avg).fillna(train["purchaseValue"].mean())

device_avg = train.groupby("deviceType")["purchaseValue"].mean()
X["device_avg_spend"] = X["deviceType"].map(device_avg).fillna(train["purchaseValue"].mean())
X_test["device_avg_spend"] = X_test["deviceType"].map(device_avg).fillna(train["purchaseValue"].mean())

X["engagement"] = X["totalHits"] + X["pageViews"] - X["totals.bounces"].fillna(0)
X["is_bounce"] = (X["totals.bounces"].fillna(0) > 0).astype(int)
X_test["engagement"] = X_test["totalHits"] + X_test["pageViews"] - X_test["totals.bounces"].fillna(0)
X_test["is_bounce"] = (X_test["totals.bounces"].fillna(0) > 0).astype(int)

X["hits_per_page"] = X["totalHits"] / (X["pageViews"] + 1e-6)
X["visit_per_session"] = X["totals.visits"] / (X["sessionNumber"] + 1e-6)
X_test["hits_per_page"] = X_test["totalHits"] / (X_test["pageViews"] + 1e-6)
X_test["visit_per_session"] = X_test["totals.visits"] / (X_test["sessionNumber"] + 1e-6)

X["hits_x_visits"] = X["totalHits"] * X["totals.visits"]
X_test["hits_x_visits"] = X_test["totalHits"] * X_test["totals.visits"]
X["bounce_ratio"] = X["totals.bounces"].fillna(0) / (X["totalHits"] + 1e-6)
X_test["bounce_ratio"] = X_test["totals.bounces"].fillna(0) / (X_test["totalHits"] + 1e-6)

for col in ["totalHits", "pageViews", "totals.visits", "hits_x_visits"]:
    X[f"log_{col}"] = np.log1p(X[col])
    X_test[f"log_{col}"] = np.log1p(X_test[col])

X["engagement_x_city_avg"] = X["engagement"] * X["city_avg_spend"]
X_test["engagement_x_city_avg"] = X_test["engagement"] * X_test["city_avg_spend"]
X["visits_x_bounce"] = X["totals.visits"] * X["is_bounce"]
X_test["visits_x_bounce"] = X_test["totals.visits"] * X_test["is_bounce"]
X["log_hits_x_visits"] = np.log1p(X["totalHits"]) * np.log1p(X["totals.visits"])
X_test["log_hits_x_visits"] = np.log1p(X_test["totalHits"]) * np.log1p(X_test["totals.visits"])

if "sessionStart" in X.columns:
    X["sessionStart"] = pd.to_datetime(X["sessionStart"], errors="coerce")
    X["hour"] = X["sessionStart"].dt.hour
    X["weekday"] = X["sessionStart"].dt.weekday
    X_test["sessionStart"] = pd.to_datetime(X_test["sessionStart"], errors="coerce")
    X_test["hour"] = X_test["sessionStart"].dt.hour
    X_test["weekday"] = X_test["sessionStart"].dt.weekday

X["channel_device"] = X["userChannel"].astype(str) + "_" + X["deviceType"].astype(str)
X_test["channel_device"] = X_test["userChannel"].astype(str) + "_" + X_test["deviceType"].astype(str)
cat_cols.append("channel_device")


In [ ]:
X['engagement_per_session'] = X['engagement'] / (X['sessionNumber'] + 1e-6)
X_test['engagement_per_session'] = X_test['engagement'] / (X_test['sessionNumber'] + 1e-6)

X['page_per_visit'] = X['pageViews'] / (X['totals.visits'] + 1e-6)
X_test['page_per_visit'] = X_test['pageViews'] / (X_test['totals.visits'] + 1e-6)

X['bounce_per_visit'] = X['totals.bounces'] / (X['totals.visits'] + 1e-6)
X_test['bounce_per_visit'] = X_test['totals.bounces'] / (X_test['totals.visits'] + 1e-6)

X['hour_bin'] = pd.cut(X['hour'], bins=[-1, 6, 12, 18, 24], labels=['Night', 'Morning', 'Afternoon', 'Evening'])
X_test['hour_bin'] = pd.cut(X_test['hour'], bins=[-1, 6, 12, 18, 24], labels=['Night', 'Morning', 'Afternoon', 'Evening'])

X['hour_bin'] = X['hour_bin'].astype(str)
X_test['hour_bin'] = X_test['hour_bin'].astype(str)
cat_cols.append('hour_bin')

X['is_weekend'] = X['weekday'].isin([5, 6]).astype(int)
X_test['is_weekend'] = X_test['weekday'].isin([5, 6]).astype(int)

session_avg_hits = train.groupby('userId')['totalHits'].mean()
X['user_avg_hits'] = X['userId'].map(session_avg_hits)
X_test['user_avg_hits'] = X_test['userId'].map(session_avg_hits)

X['z_hits'] = (X['totalHits'] - X['user_avg_hits']) / (X['user_avg_hits'].std() + 1e-6)
X_test['z_hits'] = (X_test['totalHits'] - X_test['user_avg_hits']) / (X['user_avg_hits'].std() + 1e-6)

X['hits_x_hour'] = X['totalHits'] * X['hour']
X_test['hits_x_hour'] = X_test['totalHits'] * X_test['hour']

X['bounce_rate_x_page'] = X['bounce_ratio'] * X['pageViews']
X_test['bounce_rate_x_page'] = X_test['bounce_ratio'] * X_test['pageViews']

X['device_region_combo'] = X['deviceType'].astype(str) + "_" + X['geoNetwork.region'].astype(str)
X_test['device_region_combo'] = X_test['deviceType'].astype(str) + "_" + X_test['geoNetwork.region'].astype(str)
cat_cols.append('device_region_combo')

for col in ['geoNetwork.city', 'geoNetwork.region']:
    freq = X[col].value_counts() / len(X)
    X[f'{col}_freq'] = X[col].map(freq)
    X_test[f'{col}_freq'] = X_test[col].map(freq)

X['avg_hits_per_page'] = X['totalHits'] / (X['pageViews'] + 1e-6)
X_test['avg_hits_per_page'] = X_test['totalHits'] / (X_test['pageViews'] + 1e-6)

X['bounce_per_hit'] = X['totals.bounces'] / (X['totalHits'] + 1e-6)
X_test['bounce_per_hit'] = X_test['totals.bounces'] / (X_test['totalHits'] + 1e-6)

X['channel_x_region'] = X['userChannel'].astype(str) + "_" + X['geoNetwork.region'].astype(str)
X_test['channel_x_region'] = X_test['userChannel'].astype(str) + "_" + X_test['geoNetwork.region'].astype(str)
cat_cols.append('channel_x_region')

### Dropping sessionStart column

In [ ]:
X.drop(columns=["sessionStart"], inplace=True)
X_test.drop(columns=["sessionStart"], inplace=True)

### Target encoding on categorical features

In [ ]:
cat_cols = [col for col in cat_cols if col in X.columns]
target_encoder = ce.TargetEncoder(cols=cat_cols)
X[cat_cols] = target_encoder.fit_transform(X[cat_cols], y)
X_test[cat_cols] = target_encoder.transform(X_test[cat_cols])

# Feature Selection using RFE

In [ ]:
feature_selector_model = XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

rfe = RFE(estimator=feature_selector_model, n_features_to_select=18, step=1)
rfe.fit(X, y)

selected_features = X.columns[rfe.support_].tolist()
print("Selected Features by RFE:", selected_features)

In [ ]:
X = X[selected_features]
X_test = X_test[selected_features]

# Train-Validation Split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Model Training & Hyperparameter Tuning

### ElasticNet Pipeline

In [ ]:
elasticnet_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('elasticnet', ElasticNet(random_state=42, max_iter=10000))
])

elasticnet_param_grid = {
    'elasticnet__alpha': np.logspace(-3, 2, 50),
    'elasticnet__l1_ratio': np.linspace(0, 1, 20)
}

elasticnet_search = RandomizedSearchCV(elasticnet_pipeline, elasticnet_param_grid, n_iter=30,
                                      scoring='r2', cv=3, verbose=1, n_jobs=-1, random_state=42)
elasticnet_search.fit(X_train, y_train)
print("Best ElasticNet Params:", elasticnet_search.best_params_)

y_val_pred_elastic = elasticnet_search.predict(X_val)
print("ElasticNet R2:", r2_score(y_val, y_val_pred_elastic))

### HistGradientBoostingRegressor

In [ ]:
hgb_param_grid = {
    'max_iter': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'max_leaf_nodes': [31, 50]
}

hgb = HistGradientBoostingRegressor(random_state=42)

hgb_search = RandomizedSearchCV(
    hgb, hgb_param_grid, n_iter=10, scoring='r2', cv=3, verbose=1, n_jobs=-1, random_state=42
)

hgb_search.fit(X_train, y_train)
print("Best HistGradientBoosting Params:", hgb_search.best_params_)

y_val_pred_hgb = hgb_search.predict(X_val)
print("HistGradientBoosting R2:", r2_score(y_val, y_val_pred_hgb))

### Base Learners for Stacking Regressor

In [ ]:
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='log2',
    bootstrap=False,
    random_state=42,
    n_jobs=-1
)

xgb = XGBRegressor(
    n_estimators=500,
    max_depth=9,
    learning_rate=0.1,
    colsample_bytree=0.6,
    subsample=1,
    random_state=42,
    n_jobs=-1
)

### Defining Meta Learner

In [ ]:
meta_learner = Pipeline([
    ('scale', StandardScaler()),
    ('ridge', Ridge(alpha=100, random_state=42))
]
)

### Defining Stacking Regressor

In [ ]:
stacking_model = StackingRegressor(
    estimators=[
        ('rf', rf),
        ('xgb', xgb),
    ],
    final_estimator=meta_learner,
    cv=5,
    passthrough=True,
    n_jobs=-1
)

### Fitting Stacking Model

In [ ]:
stacking_model.fit(X_train, y_train)

### Testing Stacking Model on Validation Set

In [ ]:
y_val_pred_stack = stacking_model.predict(X_val)
print("Stacking R2:", r2_score(y_val, y_val_pred_stack))

### Through testing, the best model found was Stacking Regressor

In [ ]:
best_model = stacking_model

### Training best model on complete train set

In [ ]:
best_model.fit(X, y)

### Predicting test values with the best model

In [ ]:
y_pred = best_model.predict(X_test)

### Creating final submission

In [ ]:
submission = pd.DataFrame({
    "id": range(len(y_pred)),
    "purchaseValue": y_pred
})
submission.to_csv('submission.csv', index=False)